# Full GenEval + T2I-CompBench SPFC decomposition with Ollama

This notebook generates and saves AIM-Flow spfc_decomposition_v1 files for the complete benchmark prompt sets:

- GenEval: all 553 canonical prompts across six categories.
- T2I-CompBench: all 2,400 validation prompts across color, shape, texture, 2D spatial, non-spatial, complex, numeracy, and 3D spatial categories.

The 2,953 Ollama requests run sequentially. Every accepted item is atomically checkpointed, so the job can be interrupted and safely resumed. Re-running the generation cell validates and skips cached items. Input manifests, partial progress, recorded failures, and final decompositions are saved under outputs/ollama_datasets/full_datasets/.

Use the Python (AIM-Flow) kernel. Dataset repositories are cloned only when absent, then their pinned revisions are verified. Ollama traffic remains restricted to the local loopback interface.


In [10]:
from __future__ import annotations

import hashlib
import json
import os
import re
import subprocess
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
from urllib.error import HTTPError, URLError
from urllib.parse import urlparse
from urllib.request import Request, urlopen

# Works when the kernel starts in either the repository root or notebooks/.
cwd = Path.cwd().resolve()
REPO = cwd if (cwd / "configs").is_dir() else cwd.parent
assert (REPO / "configs").is_dir(), f"Could not locate repository from {cwd}"

OLLAMA_URL = "http://127.0.0.1:11434"
MODEL = "qwen3.6:27b"
OUTPUT_DIR = REPO / "outputs" / "ollama_datasets" / "full_datasets"
MANIFEST_DIR = OUTPUT_DIR / "manifests"
TEMPERATURE = 0.20
SEED = 13
# 32K comfortably fits the full decomposition guide, examples, schema, and response.
NUM_CTX = 32768
MAX_RETRIES = 100
RETRY_SECONDS = 2
REQUEST_TIMEOUT_SECONDS = 600
KEEP_ALIVE = "30m"

# Full datasets are the default. Set a small integer only for a separate smoke-test run.
SMOKE_LIMIT_PER_DATASET: int | None = None
RUN_DATASETS = ("geneval", "t2i_compbench")
OVERWRITE = False
CONTINUE_ON_ERROR = True
MAX_CONSECUTIVE_FAILURES = 5
PROGRESS_EVERY = 10
AUTO_FETCH_DATASETS = True

GENEVAL_REPO = REPO / "external" / "GenEval"
T2I_REPO = REPO / "external" / "T2I-CompBench"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
print("Repository:", REPO)
print("Outputs:   ", OUTPUT_DIR)
print("Mode:      ", "FULL DATASETS" if SMOKE_LIMIT_PER_DATASET is None else f"SMOKE {SMOKE_LIMIT_PER_DATASET}/dataset")


Repository: /media/fezan/ASi/DVLM/steering/aim-flow
Outputs:    /media/fezan/ASi/DVLM/steering/aim-flow/outputs/ollama_datasets/full_datasets
Mode:       FULL DATASETS


## Prepare and save the complete benchmark manifests

The canonical GenEval and T2I-CompBench repositories are pinned to exact commits. Missing repositories may be cloned when AUTO_FETCH_DATASETS is enabled; existing checkouts are never changed automatically. This cell verifies all source counts and uniqueness before saving deterministic full manifests.


In [11]:
GENEVAL_REPO_URL = "https://github.com/djghosh13/geneval.git"
GENEVAL_COMMIT = "af4902f24d3ca90ebbb446dd9891a59e0f82725f"
GENEVAL_METADATA_SHA256 = "5c48e0813e812e3c373fa5c8ed07a8f0a483be30272b4427b0559c8048e67c13"
T2I_REPO_URL = "https://github.com/Karine-Huang/T2I-CompBench.git"
T2I_COMMIT = "1b7094991a57f3c22abdd4f6e8ba6c1a15517073"

T2I_CATEGORY_FILES = {
    "color": "color_val.txt",
    "shape": "shape_val.txt",
    "texture": "texture_val.txt",
    "spatial": "spatial_val.txt",
    "non_spatial": "non_spatial_val.txt",
    "complex": "complex_val.txt",
    "numeracy": "numeracy_val.txt",
    "3d_spatial": "3d_spatial_val.txt",
}
EXPECTED_GENEVAL_ITEMS = 553
EXPECTED_T2I_ITEMS_PER_CATEGORY = 300
EXPECTED_T2I_ITEMS = EXPECTED_T2I_ITEMS_PER_CATEGORY * len(T2I_CATEGORY_FILES)


def ensure_pinned_checkout(repo_dir: Path, url: str, commit: str) -> None:
    if not repo_dir.exists():
        if not AUTO_FETCH_DATASETS:
            raise FileNotFoundError(f"Missing dataset checkout: {repo_dir}")
        repo_dir.parent.mkdir(parents=True, exist_ok=True)
        print(f"Cloning {url} -> {repo_dir}")
        subprocess.run(["git", "clone", url, str(repo_dir)], check=True)
        subprocess.run(["git", "-C", str(repo_dir), "checkout", commit], check=True)
    if not (repo_dir / ".git").exists():
        raise RuntimeError(f"Dataset path is not a Git checkout: {repo_dir}")
    current = subprocess.check_output(
        ["git", "-C", str(repo_dir), "rev-parse", "HEAD"], text=True
    ).strip()
    if current != commit:
        raise RuntimeError(
            f"{repo_dir} is at {current}, expected pinned commit {commit}. "
            "Use a separate checkout at the expected revision; this notebook will not overwrite an existing checkout."
        )


def nonempty_lines(path: Path) -> list[str]:
    if not path.is_file():
        raise FileNotFoundError(path)
    return [line.strip() for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def atomic_write_json(data: dict[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    os.replace(temp, path)


def build_geneval_manifest() -> dict[str, Any]:
    metadata_path = GENEVAL_REPO / "prompts" / "evaluation_metadata.jsonl"
    digest = hashlib.sha256(metadata_path.read_bytes()).hexdigest()
    if digest != GENEVAL_METADATA_SHA256:
        raise RuntimeError(f"Unexpected GenEval metadata SHA-256: {digest}")
    records = [json.loads(line) for line in nonempty_lines(metadata_path)]
    if len(records) != EXPECTED_GENEVAL_ITEMS:
        raise ValueError(f"Expected {EXPECTED_GENEVAL_ITEMS} GenEval prompts, found {len(records)}")
    category_ranks: Counter[str] = Counter()
    samples = []
    for official_index, record in enumerate(records):
        category = str(record["tag"]).strip()
        prompt = str(record["prompt"])
        rank = category_ranks[category]
        category_ranks[category] += 1
        official_metadata = {key: value for key, value in record.items() if key not in {"prompt", "tag"}}
        samples.append(
            {
                "id": f"geneval_{category}_{rank:06d}",
                "category": category,
                "prompt": prompt,
                "source": "GenEval",
                "split": category,
                "metadata": {"official_index": official_index, **official_metadata},
            }
        )
    return {
        "benchmark": "geneval",
        "manifest_version": "1.0",
        "seed": SEED,
        "subset_size": "full",
        "source_revision": GENEVAL_COMMIT,
        "source_sha256": GENEVAL_METADATA_SHA256,
        "samples": samples,
    }


def build_t2i_compbench_manifest() -> dict[str, Any]:
    dataset_root = T2I_REPO / "examples" / "dataset"
    samples = []
    for category, filename in T2I_CATEGORY_FILES.items():
        prompts = nonempty_lines(dataset_root / filename)
        if len(prompts) != EXPECTED_T2I_ITEMS_PER_CATEGORY:
            raise ValueError(
                f"Expected {EXPECTED_T2I_ITEMS_PER_CATEGORY} prompts in {filename}, found {len(prompts)}"
            )
        for original_index, prompt in enumerate(prompts):
            samples.append(
                {
                    "id": f"t2i_{category}_{original_index:06d}",
                    "category": category,
                    "prompt": prompt,
                    "source": "T2I-CompBench",
                    "split": filename,
                    "metadata": {"original_index": original_index},
                }
            )
    if len(samples) != EXPECTED_T2I_ITEMS:
        raise ValueError(f"Expected {EXPECTED_T2I_ITEMS} T2I-CompBench prompts, found {len(samples)}")
    return {
        "benchmark": "t2i_compbench",
        "manifest_version": "1.0",
        "seed": SEED,
        "subset_size": "full",
        "source_revision": T2I_COMMIT,
        "category_files": T2I_CATEGORY_FILES,
        "samples": samples,
    }


def validate_source_manifest(manifest: dict[str, Any], expected_count: int) -> None:
    samples = manifest["samples"]
    if len(samples) != expected_count:
        raise ValueError(f"Expected {expected_count} samples, found {len(samples)}")
    ids = [sample["id"] for sample in samples]
    prompts = [sample["prompt"] for sample in samples]
    if len(ids) != len(set(ids)):
        raise ValueError(f"Duplicate IDs in {manifest['benchmark']} manifest")
    if len(prompts) != len(set(prompts)):
        raise ValueError(f"Duplicate prompts in {manifest['benchmark']} manifest")


ensure_pinned_checkout(GENEVAL_REPO, GENEVAL_REPO_URL, GENEVAL_COMMIT)
ensure_pinned_checkout(T2I_REPO, T2I_REPO_URL, T2I_COMMIT)

MANIFESTS = {
    "geneval": build_geneval_manifest(),
    "t2i_compbench": build_t2i_compbench_manifest(),
}
validate_source_manifest(MANIFESTS["geneval"], EXPECTED_GENEVAL_ITEMS)
validate_source_manifest(MANIFESTS["t2i_compbench"], EXPECTED_T2I_ITEMS)

MANIFEST_PATHS = {
    name: MANIFEST_DIR / f"{name}_full_seed{SEED}.json"
    for name in MANIFESTS
}
for name, manifest in MANIFESTS.items():
    atomic_write_json(manifest, MANIFEST_PATHS[name])
    counts = Counter(sample["category"] for sample in manifest["samples"])
    print(f"{name}: {len(manifest['samples'])} prompts -> {MANIFEST_PATHS[name]}")
    print("  categories:", dict(counts))
print("Total prompts:", sum(len(manifest["samples"]) for manifest in MANIFESTS.values()))


geneval: 553 prompts -> /media/fezan/ASi/DVLM/steering/aim-flow/outputs/ollama_datasets/full_datasets/manifests/geneval_full_seed13.json
  categories: {'single_object': 80, 'two_object': 99, 'counting': 80, 'colors': 94, 'position': 100, 'color_attr': 100}
t2i_compbench: 2400 prompts -> /media/fezan/ASi/DVLM/steering/aim-flow/outputs/ollama_datasets/full_datasets/manifests/t2i_compbench_full_seed13.json
  categories: {'color': 300, 'shape': 300, 'texture': 300, 'spatial': 300, 'non_spatial': 300, 'complex': 300, 'numeracy': 300, '3d_spatial': 300}
Total prompts: 2953


## Verify the local Ollama runtime

Only loopback Ollama URLs are accepted. Start Ollama first with ollama serve if this check cannot connect. The configured model must already be pulled; the notebook never downloads an Ollama model.


In [12]:
def local_ollama_request(path: str, payload: dict[str, Any] | None = None) -> dict[str, Any]:
    parsed = urlparse(OLLAMA_URL)
    if parsed.scheme != "http" or parsed.hostname not in {"127.0.0.1", "localhost", "::1"}:
        raise ValueError("OLLAMA_URL must be a local, plain-HTTP loopback address")
    body = None if payload is None else json.dumps(payload).encode("utf-8")
    request = Request(
        OLLAMA_URL.rstrip("/") + path,
        data=body,
        headers={"Content-Type": "application/json"},
        method="GET" if payload is None else "POST",
    )
    try:
        with urlopen(request, timeout=REQUEST_TIMEOUT_SECONDS) as response:
            return json.loads(response.read().decode("utf-8"))
    except (HTTPError, URLError, TimeoutError) as exc:
        raise RuntimeError(f"Local Ollama request failed ({path}): {exc}") from exc


tags = local_ollama_request("/api/tags").get("models", [])
installed = sorted(model.get("name", "") for model in tags)
if MODEL not in installed:
    raise RuntimeError(f"Required local model {MODEL!r} is not installed. Installed: {installed}")
print("Ready - local model found:", MODEL)
print("Network target:", OLLAMA_URL, "(loopback only)")
print("Ollama request context:", NUM_CTX)


Ready - local model found: qwen3.6:27b
Network target: http://127.0.0.1:11434 (loopback only)
Ollama request context: 32768


## Full decomposition instruction and strict output validation

The system prompt below is the full AIM-Flow/SPFC decomposition guide. Ollama structured output and local validation both enforce the repository schema, sequential names, exact final-target copying, and primitive limits.


In [13]:
ITEM_SCHEMA = {
    "type": "object",
    "properties": {
        "source_prompt": {"type": "string"},
        "negative_prompt": {"type": "string"},
        "primitive_prompts": {
            "type": "array",
            "minItems": 2,
            "maxItems": 8,
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "text": {"type": "string"},
                    "role": {"type": "string", "enum": ["primitive"]},
                    "weight": {"type": "number", "minimum": 0, "maximum": 2},
                    "enabled": {"type": "boolean"}
                },
                "required": ["name", "text", "role", "weight", "enabled"],
                "additionalProperties": False
            }
        }
    },
    "required": ["source_prompt", "negative_prompt", "primitive_prompts"],
    "additionalProperties": False
}

SYSTEM_PROMPT = """You create high-quality prompt decompositions for AIM-Flow/SPFC text-to-image generation.

Return only the JSON object required by the supplied schema. Do not include explanations, Markdown, or extra keys.

## Objective

Decompose the supplied target prompt into:

* A minimally ablated `source_prompt`
* Focused `primitives`
* A concise, target-specific `negative_prompt`

The decomposition should teach a diffusion model the individual entities, attribute bindings, counts, and relations before reconstructing the complete target.

## 1. Parse the target

Silently identify:

* **Entities:** objects, people, animals, surfaces, and containers
* **Attributes:** colour, material, shape, texture, size, clothing, and state
* **Bindings:** which attribute belongs to which entity
* **Counts:** exact number of each entity
* **Relations:** left/right, above/below, on top of/underneath, in front of/behind, inside/containing, beside, near, between, holding, wearing, reflected in, and similar relations
* **Global descriptors:** style, lighting, medium, camera, and quality terms

Preserve every explicit constraint. Do not swap attributes, reverse relations, or invent unstated details.

## 2. Source prompt

`source_prompt` must be a minimal, non-contradictory ablation of the target.

* Preserve the target’s entities and neutral scene context.
* Preserve relevant style, lighting, medium, and quality terms.
* Remove the attributes, counts, or relations that the primitives are intended to introduce.
* Do not replace removed details with contradictory details.
* For simple two-entity relation prompts, list both entities without asserting the tested relation.

Example:

Target:
`A television on the right of a woman.`

Source:
`A television and a woman.`

Do not introduce a substitute relation such as “near,” “beside,” or “together” unless it is explicitly present in the target.

## 3. Primitive construction

Use 2–8 primitives in total, including the final complete-target primitive.

Create primitives in this order when applicable:

1. Entity identities
2. Attribute-bound entities
3. Exact counts
4. Primary relations
5. Useful inverse relations
6. Complete target

Each primitive must be concise, standalone, visually grounded, and renderable.

### Entity primitives

Use separate identity primitives when they provide useful grounding.

Examples:

* `An airplane.`
* `A mouse.`

Do not add unnecessary identity primitives when the entity is already clearly established by a more informative attribute-bound primitive.

### Attribute primitives

Bind every attribute explicitly to its correct entity.

Correct:

* `A red cube.`
* `A woman wearing a blue shirt.`
* `A cat made of glass.`

Avoid:

* `A cube and red.`
* `A woman and a blue shirt.`
* `Glass and a cat.`

When multiple entities have attributes, create separate primitives when needed to prevent colour, material, or clothing swaps.

### Count primitives

Preserve exact counts explicitly.

Correct:

* `Exactly three yellow apples.`
* `Two dogs and one cat.`

Do not replace exact counts with vague plurals.

### Relation primitives

Every relation primitive must include:

* The subject
* The complete relation
* The reference entity or destination

Correct:

* `A television positioned to the right of a woman.`
* `An airplane positioned above a mouse.`
* `Three apples contained inside a bowl.`

Avoid:

* `A television on the right.`
* `An airplane and a mouse.`
* `Apples with a bowl.`

Preserve the relation direction exactly. Interpret left and right from the image viewer’s perspective unless the target explicitly specifies another viewpoint.

Do not strengthen the relation with words such as `directly`, `touching`, `attached`, `centered`, or `perfectly` unless the target requires them.

## 4. Inverse-relation policy

Add an inverse-relation primitive only when it provides a distinct directional description of the same scene.

Use inverse primitives for asymmetric relation pairs such as:

* `X left of Y` → `Y right of X`
* `X right of Y` → `Y left of X`
* `X above Y` → `Y below X`
* `X on top of Y` → `Y underneath X`
* `X in front of Y` → `Y behind X`
* `X inside Y` → `Y containing X`

Example:

* `An airplane positioned above a mouse.`
* `A mouse positioned underneath an airplane.`

Do not add inverse primitives for symmetric or nearly symmetric relations, because they provide duplicate supervision.

Do not invert:

* beside
* next to
* near
* adjacent to
* overlapping with

For example, these are duplicates and should not both appear:

* `A fish beside a bag.`
* `A bag beside a fish.`

Use only one clear relation primitive for symmetric relations.

## 5. Ambiguous or awkward target wording

The final primitive must preserve the target exactly, but focused primitives may use clearer grammatical wording.

Choose the most literal visually plausible interpretation without inventing extra meaning.

Example:

Target:
`a fish on side of a bag`

Acceptable relation primitive:
`A fish positioned at the side of a bag.`

Do not reinterpret it as:

* a fish inside the bag
* a fish printed on the bag
* a fish attached to the bag
* a fish on top of the bag

unless the target explicitly states that meaning.

When the relation remains ambiguous, preserve its general meaning rather than making it more specific.

## 6. Complex effects

Reflections, shadows, occlusions, and unusual interactions must remain complete visual events.

Correct:

* `A fox statue reflected in a cracked mirror.`
* `The shadow cast by a fox statue forms the shape of a dragon on the wall.`

Avoid:

* `A reflected fox.`
* `A dragon shadow.`
* `A dragon on the wall.`

Separate the underlying object, its special attribute, and the complex effect when each is an important target constraint.

## 7. Avoid redundant primitives

Every primitive must add a distinct supervision signal.

Do not create:

* Two paraphrases of the same symmetric relation
* A generic object primitive followed by an almost identical object primitive
* Both a relation and its inverse when the inverse adds no directional information
* Multiple primitives that differ only through minor wording

Prefer fewer strong primitives over redundant ones.

## 8. Final primitive

The final primitive must:

* Be an exact character-for-character copy of the supplied target prompt
* Appear last
* Have weight `1.1`
* Represent the complete target scene

Do not correct its spelling, grammar, capitalization, punctuation, or wording.

All preceding primitives should normally have weight `1.0`.

## 9. Naming and fixed fields

Primitive names must be unique snake_case identifiers prefixed sequentially:

* `S1_...`
* `S2_...`
* `S3_...`

Every primitive must have:

* `"role": "primitive"`
* `"enabled": true`

## 10. Negative prompt

Treat `negative_prompt` as a light diffusion-conditioning signal, not as a logical explanation.

Use a concise comma-separated list of approximately 6–14 visible failure modes.

Prioritize:

* Missing required entities
* Wrong entity identity
* Wrong material, colour, shape, or clothing
* Attributes attached to the wrong entity
* Incorrect count
* One or two likely incorrect spatial configurations
* Extra salient objects
* Blur, distortion, malformed objects
* Low resolution
* Text, logos, watermarks

Describe concrete unwanted images rather than abstract reasoning.

Prefer:

* `airplane below mouse`
* `mouse above airplane`
* `red sphere`
* `blue cube`
* `fish inside bag`
* `four apples`

Avoid:

* `incorrect spatial relation`
* `reversed relation`
* `wrong composition`
* `object not correctly positioned`
* long logical sentences
* many paraphrases of the same failure

Do not include every possible wrong relation. Include only the most likely confusions.

Never negate anything required by the target.

## Examples

### Example 1: Asymmetric spatial relation

Target:
`an airplane on top of a mouse`

Output:
{
"source_prompt": "An airplane and a mouse.",
"primitive_prompts": [
{
"name": "S1_airplane",
"text": "An airplane.",
"weight": 1.0,
"role": "primitive",
"enabled": true
},
{
"name": "S2_mouse",
"text": "A mouse.",
"weight": 1.0,
"role": "primitive",
"enabled": true
},
{
"name": "S3_airplane_above_mouse",
"text": "An airplane positioned above a mouse.",
"weight": 1.0,
"role": "primitive",
"enabled": true
},
{
"name": "S4_mouse_underneath_airplane",
"text": "A mouse positioned underneath an airplane.",
"weight": 1.0,
"role": "primitive",
"enabled": true
},
{
"name": "S5_complete_target",
"text": "an airplane on top of a mouse",
"weight": 1.1,
"role": "primitive",
"enabled": true
}
],
"negative_prompt": "missing airplane, missing mouse, airplane below mouse, mouse above airplane, extra salient objects, malformed airplane, malformed mouse, blur, low resolution, text, watermarks"
}

### Example 2: Symmetric relation

Target:
`a fish on side of a bag`

Output:
{
"source_prompt": "A fish and a bag.",
"primitive_prompts": [
{
"name": "S1_fish",
"text": "A fish.",
"weight": 1.0,
"role": "primitive",
"enabled": true
},
{
"name": "S2_bag",
"text": "A bag.",
"weight": 1.0,
"role": "primitive",
"enabled": true
},
{
"name": "S3_fish_at_side_of_bag",
"text": "A fish positioned at the side of a bag.",
"weight": 1.0,
"role": "primitive",
"enabled": true
},
{
"name": "S4_complete_target",
"text": "a fish on side of a bag",
"weight": 1.1,
"role": "primitive",
"enabled": true
}
],
"negative_prompt": "missing fish, missing bag, fish inside bag, fish above bag, fish underneath bag, extra salient objects, malformed fish, malformed bag, blur, low resolution, text, watermarks"
}

### Example 3: Colour binding and direction

Target:
`A red cube to the left of a blue sphere.`

Output:
{
"source_prompt": "A cube and a sphere.",
"primitive_prompts": [
{
"name": "S1_red_cube",
"text": "A red cube.",
"weight": 1.0,
"role": "primitive",
"enabled": true
},
{
"name": "S2_blue_sphere",
"text": "A blue sphere.",
"weight": 1.0,
"role": "primitive",
"enabled": true
},
{
"name": "S3_red_cube_left_of_blue_sphere",
"text": "A red cube positioned to the left of a blue sphere.",
"weight": 1.0,
"role": "primitive",
"enabled": true
},
{
"name": "S4_blue_sphere_right_of_red_cube",
"text": "A blue sphere positioned to the right of a red cube.",
"weight": 1.0,
"role": "primitive",
"enabled": true
},
{
"name": "S5_complete_target",
"text": "A red cube to the left of a blue sphere.",
"weight": 1.1,
"role": "primitive",
"enabled": true
}
],
"negative_prompt": "missing cube, missing sphere, blue cube, red sphere, cube right of sphere, sphere left of cube, extra salient objects, distorted shapes, blur, low resolution, text, watermarks"
}

### Example 4: Count and containment

Target:
`Three yellow apples inside a blue bowl.`

Output:
{
"source_prompt": "Apples and a bowl.",
"primitive_prompts": [
{
"name": "S1_three_yellow_apples",
"text": "Exactly three yellow apples.",
"weight": 1.0,
"role": "primitive",
"enabled": true
},
{
"name": "S2_blue_bowl",
"text": "A blue bowl.",
"weight": 1.0,
"role": "primitive",
"enabled": true
},
{
"name": "S3_apples_inside_bowl",
"text": "Exactly three yellow apples contained inside a blue bowl.",
"weight": 1.0,
"role": "primitive",
"enabled": true
},
{
"name": "S4_bowl_containing_apples",
"text": "A blue bowl containing exactly three yellow apples.",
"weight": 1.0,
"role": "primitive",
"enabled": true
},
{
"name": "S5_complete_target",
"text": "Three yellow apples inside a blue bowl.",
"weight": 1.1,
"role": "primitive",
"enabled": true
}
],
"negative_prompt": "missing apples, missing bowl, two apples, four apples, non-yellow apples, non-blue bowl, apples outside bowl, apples beside bowl, extra fruit, blur, low resolution, text, watermarks"
}

Before returning the JSON, silently verify:

* Every required entity is represented.
* Every attribute is bound to the correct entity.
* Exact counts are preserved.
* Relations retain the correct subject, reference, and direction.
* Inverse relations are used only for asymmetric relations.
* Symmetric relations are not duplicated.
* Ambiguous wording has not been overinterpreted.
* Every primitive contributes distinct supervision.
* Negative phrases describe concrete visible failures.
* The source does not contradict the target.
* The final primitive exactly matches the supplied target.
"""

SYSTEM_PROMPT_SHA256 = hashlib.sha256(SYSTEM_PROMPT.encode("utf-8")).hexdigest()

def user_prompt(sample: dict[str, Any], feedback: str | None = None) -> str:
    text = (
        f"Dataset: {sample['source']}\n"
        f"Category: {sample['category']}\n"
        f"Target prompt: {sample['prompt']}\n"
        f"Metadata: {json.dumps(sample.get('metadata', {}), sort_keys=True)}\n"
        "Create its SPFC decomposition now."
    )
    return text if not feedback else text + f"\nYour previous result was invalid: {feedback}\nCorrect it."

def parse_json_object(text: str) -> dict[str, Any]:
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.I | re.S).strip()
    value = json.loads(text)
    if not isinstance(value, dict):
        raise ValueError("model response is not a JSON object")
    return value

def validate_generated(sample: dict[str, Any], generated: dict[str, Any]) -> dict[str, Any]:
    allowed = {"source_prompt", "negative_prompt", "primitive_prompts"}
    if set(generated) != allowed:
        raise ValueError(f"expected keys {sorted(allowed)}, got {sorted(generated)}")
    source = str(generated["source_prompt"]).strip()
    negative = str(generated["negative_prompt"]).strip()
    primitives = generated["primitive_prompts"]
    if not source or not negative:
        raise ValueError("source_prompt and negative_prompt must be non-empty")
    if not isinstance(primitives, list) or not 2 <= len(primitives) <= 8:
        raise ValueError("primitive_prompts must contain 2-8 items")
    names = []
    for index, primitive in enumerate(primitives, 1):
        if set(primitive) != {"name", "text", "role", "weight", "enabled"}:
            raise ValueError(f"primitive {index} has unexpected fields")
        name, text = str(primitive["name"]).strip(), str(primitive["text"]).strip()
        if not re.fullmatch(rf"S{index}_[a-z0-9_]+", name):
            raise ValueError(f"primitive {index} name must match S{index}_snake_case")
        if not text or primitive["role"] != "primitive" or primitive["enabled"] is not True:
            raise ValueError(f"primitive {index} has invalid text, role, or enabled value")
        weight = float(primitive["weight"])
        if not 0 <= weight <= 2:
            raise ValueError(f"primitive {index} weight is outside [0, 2]")
        primitive.update(name=name, text=text, weight=weight)
        names.append(name)
    if len(names) != len(set(names)):
        raise ValueError("primitive names must be unique")
    if primitives[-1]["text"] != sample["prompt"]:
        raise ValueError("final primitive text must exactly match the target prompt")
    if abs(float(primitives[-1]["weight"]) - 1.1) > 1e-9:
        raise ValueError("final primitive weight must be 1.1")
    return {
        "id": sample["id"],
        "target_prompt": sample["prompt"].strip(),
        "source_prompt": source,
        "negative_prompt": negative,
        "primitive_prompts": primitives,
        "metadata": {
            **sample.get("metadata", {}),
            "category": sample["category"],
            "template": False,
            "generator": "ollama",
            "model": MODEL,
            "num_ctx": NUM_CTX,
            "system_prompt_sha256": SYSTEM_PROMPT_SHA256,
        },
    }

def generate_item(sample: dict[str, Any]) -> dict[str, Any]:
    feedback = None
    for attempt in range(1, MAX_RETRIES + 1):
        payload = {
            "model": MODEL,
            "stream": False,
            "think": False,
            "format": ITEM_SCHEMA,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt(sample, feedback)},
            ],
            "options": {"temperature": TEMPERATURE, "seed": SEED, "num_ctx": NUM_CTX},
            "keep_alive": "30m",
        }
        try:
            response = local_ollama_request("/api/chat", payload)
            generated = parse_json_object(response["message"]["content"])
            return validate_generated(sample, generated)
        except Exception as exc:
            feedback = f"{type(exc).__name__}: {exc}"
            if attempt == MAX_RETRIES:
                raise RuntimeError(f"Failed {sample['id']} after {MAX_RETRIES} attempts: {feedback}") from exc
            print(f"  retry {attempt}/{MAX_RETRIES - 1}: {feedback}")
            time.sleep(RETRY_SECONDS)
    raise AssertionError("unreachable")

print("Generation and validation helpers loaded.")


Generation and validation helpers loaded.


## Generate both complete decomposition datasets

The cell below processes GenEval first and T2I-CompBench second. Accepted items are atomically saved after every request. Cached valid items are skipped on rerun; recorded failures are retried. A separate filename is used whenever SMOKE_LIMIT_PER_DATASET is set, so smoke tests cannot contaminate full outputs.


In [15]:
def load_manifest(path: Path) -> dict[str, Any]:
    manifest = json.loads(path.read_text(encoding="utf-8"))
    samples = manifest.get("samples")
    if manifest.get("subset_size") != "full":
        raise ValueError(f"Expected a full manifest in {path}")
    if not isinstance(samples, list) or not samples:
        raise ValueError(f"No samples in {path}")
    ids = [sample.get("id") for sample in samples]
    if len(ids) != len(set(ids)):
        raise ValueError(f"Duplicate sample IDs in {path}")
    return manifest


def selected_samples(manifest: dict[str, Any]) -> list[dict[str, Any]]:
    samples = manifest["samples"]
    if SMOKE_LIMIT_PER_DATASET is None:
        return samples
    if SMOKE_LIMIT_PER_DATASET < 1:
        raise ValueError("SMOKE_LIMIT_PER_DATASET must be positive or None")
    return samples[:SMOKE_LIMIT_PER_DATASET]


def output_path_for(name: str) -> Path:
    suffix = "" if SMOKE_LIMIT_PER_DATASET is None else f"_smoke_{SMOKE_LIMIT_PER_DATASET}"
    return OUTPUT_DIR / f"{name}_full_seed{SEED}{suffix}_spfc_ollama.json"


def output_state(
    name: str,
    manifest: dict[str, Any],
    samples: list[dict[str, Any]],
    completed: dict[str, dict[str, Any]],
    failures: dict[str, dict[str, Any]],
) -> dict[str, Any]:
    ordered = [completed[sample["id"]] for sample in samples if sample["id"] in completed]
    return {
        "schema_version": "spfc_decomposition_v1",
        "benchmark": name,
        "subset_size": "full" if SMOKE_LIMIT_PER_DATASET is None else len(samples),
        "seed": SEED,
        "expected_items": len(samples),
        "completed_items": len(ordered),
        "source_manifest": str(MANIFEST_PATHS[name]),
        "source_revision": manifest.get("source_revision"),
        "generation": {
            "generator": "ollama",
            "model": MODEL,
            "num_ctx": NUM_CTX,
            "temperature": TEMPERATURE,
            "system_prompt_sha256": SYSTEM_PROMPT_SHA256,
            "updated_at": datetime.now(timezone.utc).isoformat(),
        },
        "failures": failures,
        "items": ordered,
    }


def save_progress(
    name: str,
    manifest: dict[str, Any],
    samples: list[dict[str, Any]],
    completed: dict[str, dict[str, Any]],
    failures: dict[str, dict[str, Any]],
    output_path: Path,
) -> None:
    atomic_write_json(output_state(name, manifest, samples, completed, failures), output_path)


def load_resumable_items(
    output_path: Path,
    samples_by_id: dict[str, dict[str, Any]],
) -> tuple[dict[str, dict[str, Any]], dict[str, dict[str, Any]]]:
    if OVERWRITE or not output_path.exists():
        return {}, {}
    saved = json.loads(output_path.read_text(encoding="utf-8"))
    if saved.get("schema_version") != "spfc_decomposition_v1":
        raise ValueError(f"Cannot resume unsupported schema in {output_path}")
    valid: dict[str, dict[str, Any]] = {}
    for item in saved.get("items", []):
        sample = samples_by_id.get(item.get("id"))
        if sample is None:
            continue
        try:
            generated = {
                key: item[key]
                for key in ("source_prompt", "negative_prompt", "primitive_prompts")
            }
            valid[item["id"]] = validate_generated(sample, generated)
        except Exception as exc:
            print(f"Dropping invalid cached item {item.get('id')}: {type(exc).__name__}: {exc}")
    failures = {
        sample_id: details
        for sample_id, details in dict(saved.get("failures") or {}).items()
        if sample_id in samples_by_id and sample_id not in valid
    }
    return valid, failures


def generate_dataset(name: str, manifest_path: Path) -> Path:
    manifest = load_manifest(manifest_path)
    samples = selected_samples(manifest)
    samples_by_id = {sample["id"]: sample for sample in samples}
    output_path = output_path_for(name)
    completed, failures = load_resumable_items(output_path, samples_by_id)
    pending = [sample for sample in samples if sample["id"] not in completed]
    print(
        f"\n{name}: {len(samples)} required; {len(completed)} cached; "
        f"{len(pending)} pending; {len(failures)} prior failures"
    )
    print("Output:", output_path)

    started = time.monotonic()
    attempted = 0
    consecutive_failures = 0
    for overall_index, sample in enumerate(samples, 1):
        if sample["id"] in completed:
            continue
        attempted += 1
        print(f"[{overall_index:04d}/{len(samples):04d}] {sample['id']} - {sample['prompt']}")
        try:
            completed[sample["id"]] = generate_item(sample)
        except KeyboardInterrupt:
            save_progress(name, manifest, samples, completed, failures, output_path)
            print(f"Interrupted safely after {len(completed)}/{len(samples)} items.")
            raise
        except Exception as exc:
            consecutive_failures += 1
            failures[sample["id"]] = {
                "error": f"{type(exc).__name__}: {exc}",
                "failed_at": datetime.now(timezone.utc).isoformat(),
            }
            save_progress(name, manifest, samples, completed, failures, output_path)
            print(f"  FAILED ({consecutive_failures} consecutive): {failures[sample['id']]['error']}")
            if not CONTINUE_ON_ERROR:
                raise
            if consecutive_failures >= MAX_CONSECUTIVE_FAILURES:
                raise RuntimeError(
                    f"Stopping after {MAX_CONSECUTIVE_FAILURES} consecutive failures. "
                    "Progress is saved; fix the runtime issue and rerun this cell."
                ) from exc
            continue

        consecutive_failures = 0
        failures.pop(sample["id"], None)
        save_progress(name, manifest, samples, completed, failures, output_path)

        if attempted % PROGRESS_EVERY == 0 or len(completed) == len(samples):
            elapsed = time.monotonic() - started
            average = elapsed / attempted
            remaining = len(samples) - len(completed)
            eta_hours = average * remaining / 3600
            print(
                f"  checkpoint: {len(completed)}/{len(samples)} complete, "
                f"{len(failures)} failures, ETA {eta_hours:.1f} h"
            )

    save_progress(name, manifest, samples, completed, failures, output_path)
    elapsed = time.monotonic() - started
    print(
        f"Finished pass for {name}: {len(completed)}/{len(samples)} complete, "
        f"{len(failures)} failures, {elapsed / 60:.1f} min this run"
    )
    if failures:
        print("Re-run this cell to retry failed items; valid items will remain cached.")
    return output_path


generated_paths = {
    name: generate_dataset(name, MANIFEST_PATHS[name])
    for name in RUN_DATASETS
}
generated_paths



geneval: 553 required; 553 cached; 0 pending; 0 prior failures
Output: /media/fezan/ASi/DVLM/steering/aim-flow/outputs/ollama_datasets/full_datasets/geneval_full_seed13_spfc_ollama.json
Finished pass for geneval: 553/553 complete, 0 failures, 0.0 min this run

t2i_compbench: 2400 required; 2400 cached; 0 pending; 0 prior failures
Output: /media/fezan/ASi/DVLM/steering/aim-flow/outputs/ollama_datasets/full_datasets/t2i_compbench_full_seed13_spfc_ollama.json
Finished pass for t2i_compbench: 2400/2400 complete, 0 failures, 0.0 min this run


{'geneval': PosixPath('/media/fezan/ASi/DVLM/steering/aim-flow/outputs/ollama_datasets/full_datasets/geneval_full_seed13_spfc_ollama.json'),
 't2i_compbench': PosixPath('/media/fezan/ASi/DVLM/steering/aim-flow/outputs/ollama_datasets/full_datasets/t2i_compbench_full_seed13_spfc_ollama.json')}

## Final completeness and repository-schema validation

This cell passes only when every selected prompt has a valid decomposition. If a long run has recorded failures, rerun the generation cell first; all valid cached items will be preserved.


In [16]:
from aim_flow.eval_bench.schemas import DecompositionManifest

incomplete = []
for name, path in generated_paths.items():
    manifest = load_manifest(MANIFEST_PATHS[name])
    samples = selected_samples(manifest)
    saved = json.loads(path.read_text(encoding="utf-8"))
    completed = int(saved.get("completed_items", len(saved.get("items", []))))
    failures = dict(saved.get("failures") or {})
    if completed != len(samples) or failures:
        incomplete.append(
            f"{name}: {completed}/{len(samples)} complete, {len(failures)} failures"
        )
        continue
    result = DecompositionManifest.load(path)
    if len(result.items) != len(samples):
        raise ValueError(f"{name}: schema loaded {len(result.items)} items, expected {len(samples)}")
    expected_targets = {sample["id"]: sample["prompt"] for sample in samples}
    for item in result.items:
        if item.target_prompt != expected_targets[item.id]:
            raise ValueError(f"{name}: target mismatch for {item.id}")
    print(f"PASS: {path.name}: {len(result.items)} complete, schema-valid items")

if incomplete:
    raise RuntimeError(
        "Full decomposition outputs are not complete yet. Re-run the generation cell.\n"
        + "\n".join(incomplete)
    )
print("PASS: every requested GenEval and T2I-CompBench prompt is saved and validated.")


PASS: geneval_full_seed13_spfc_ollama.json: 553 complete, schema-valid items
PASS: t2i_compbench_full_seed13_spfc_ollama.json: 2400 complete, schema-valid items
PASS: every requested GenEval and T2I-CompBench prompt is saved and validated.
